### Load Processed Data

In [1]:
import pandas as pd
import numpy as np

In [2]:
#loading interaction matrix

interaction_matrix = pd.read_pickle(
    "../data/processed/interaction_matrix.pkl"
)


In [3]:
interaction_matrix.head()

movie_id,1,2,3,4,5,6,7,8,9,10,...,1673,1674,1675,1676,1677,1678,1679,1680,1681,1682
user_id,,,,,,,,,,,,,,,,,,,,,
1,5.0,3.0,4.0,3.0,3.0,5.0,4.0,1.0,5.0,3.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,4.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,4.0,3.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


# User -User Collaborative Filtering



In [4]:
#calculating user similarity

In [5]:
from sklearn.metrics.pairwise import cosine_similarity

In [6]:
interaction_filled = interaction_matrix.fillna(0)

user_similarity = cosine_similarity(
    interaction_filled
)

In [7]:
user_similarity_df = pd.DataFrame(
    user_similarity,
    index=interaction_matrix.index,
    columns=interaction_matrix.index
)

In [8]:
user_similarity_df.iloc[:5,:5]

user_id,1,2,3,4,5
user_id,,,,,
1,1.000000,0.166931,0.047460,0.064358,0.378475
2,0.166931,1.000000,0.110591,0.178121,0.072979
3,0.047460,0.110591,1.000000,0.344151,0.021245
4,0.064358,0.178121,0.344151,1.000000,0.031804
5,0.378475,0.072979,0.021245,0.031804,1.000000


In [9]:
def get_similar_users(user_id,top_n=5):
    similar_users=user_similarity_df[user_id].sort_values(ascending=False)

    return similar_users.iloc[1:top_n+1]

In [10]:
get_similar_users(1,6)

user_id
916    0.569066
864    0.547548
268    0.542077
92     0.540534
435    0.538665
457    0.538476
Name: 1, dtype: float64

## Recommend movies from similar users


In [11]:
ratings = pd.read_parquet(
    "../data/processed/ratings_features.parquet"
)

movies = pd.read_parquet(
    "../data/processed/movies_features.parquet"
)

In [12]:
ratings.head()

,user_id,movie_id,rating,timestamp,user_mean,normalised_rating
0,196,242,3,881250949,3.615385,-0.615385
1,186,302,3,891717742,3.413043,-0.413043
2,22,377,1,878887116,3.351562,-2.351562
3,244,51,2,880606923,3.651261,-1.651261
4,166,346,1,886397596,3.550000,-2.550000


In [13]:
movies.head()

,movie_id,title,unknown,Action,Adventure,Animation,Children,Comedy,Crime,Documentary,...,Mystery,Romance,Sci-Fi,Thriller,War,Western,genres_test,genres_text,num_genres,release_year
0,1,Toy Story (1995),0,0,0,1,1,1,0,0,...,0,0,0,0,0,0,Animation Children Comedy,Animation Children Comedy,3,1995.0
1,2,GoldenEye (1995),0,1,1,0,0,0,0,0,...,0,0,0,1,0,0,Action Adventure Thriller,Action Adventure Thriller,3,1995.0
2,3,Four Rooms (1995),0,0,0,0,0,0,0,0,...,0,0,0,1,0,0,Thriller,Thriller,1,1995.0
3,4,Get Shorty (1995),0,1,0,0,0,1,0,0,...,0,0,0,0,0,0,Action Comedy Drama,Action Comedy Drama,3,1995.0
4,5,Copycat (1995),0,0,0,0,0,0,1,0,...,0,0,0,1,0,0,Crime Drama Thriller,Crime Drama Thriller,3,1995.0


In [14]:
# Find similar users
#       ↓
# Look at movies they liked
#       ↓
# Remove movies already watched
#       ↓
# Recommend remaining movies

In [15]:
def recommend_user_user(user_id,top_n=10):

    similar_users=get_similar_users(user_id)

    similar_user_ids=similar_users.index

    watched_movies=set(ratings[ratings.user_id==user_id]['movie_id'])

    recommendations={}

    for sim_user in similar_user_ids:
        user_movies=ratings[ratings.user_id==sim_user] #movies watched by similar user
        #print(user_movies)

        for _,row in user_movies.iterrows():
            movie=row.movie_id

            if movie not in watched_movies:
                recommendations[movie]=(
                    recommendations.get(movie,0)
                    +row.rating
                )

    #sort by score
    recs=sorted(
        recommendations.items(),
        key=lambda x:x[1],
        reverse=True
    )

    return recs[:top_n]
        

# Item Item Collaborative filtering

In [16]:
# those who liked Batman will also like Dark Night then recommend dark night to another user

In [17]:
interaction_matrix.head()

movie_id,1,2,3,4,5,6,7,8,9,10,...,1673,1674,1675,1676,1677,1678,1679,1680,1681,1682
user_id,,,,,,,,,,,,,,,,,,,,,
1,5.0,3.0,4.0,3.0,3.0,5.0,4.0,1.0,5.0,3.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,4.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,4.0,3.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [18]:
movie_matrix=interaction_matrix.T

In [19]:
movie_matrix.head()

user_id,1,2,3,4,5,6,7,8,9,10,...,934,935,936,937,938,939,940,941,942,943
movie_id,,,,,,,,,,,,,,,,,,,,,
1,5.0,4.0,NaN,NaN,4.0,4.0,NaN,NaN,NaN,4.0,...,2.0,3.0,4.0,NaN,4.0,NaN,NaN,5.0,NaN,NaN
2,3.0,NaN,NaN,NaN,3.0,NaN,NaN,NaN,NaN,NaN,...,4.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5.0
3,4.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,4.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,3.0,NaN,NaN,NaN,NaN,NaN,5.0,NaN,NaN,4.0,...,5.0,NaN,NaN,NaN,NaN,NaN,2.0,NaN,NaN,NaN
5,3.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [20]:
movie_matrix_filled = movie_matrix.fillna(0)

In [21]:
movie_similarity=cosine_similarity(movie_matrix_filled)

movie_similarity_df=pd.DataFrame(
    movie_similarity,
    index=movie_matrix.index,
    columns=movie_matrix.index
)

In [22]:
#find similar movies

def similar_movies(movie_id,top_n=10):
    sims=movie_similarity_df[movie_id]

    return sims.sort_values(
        ascending=False
    )[1:top_n+1]

In [23]:
similar_movies(50)

movie_id
181    0.884476
174    0.764885
172    0.749819
1      0.734572
127    0.697332
121    0.692837
210    0.689343
100    0.686533
98     0.676428
222    0.673975
Name: 50, dtype: float64

### Item  Recommendation for User



In [24]:
# Look at movies user liked.

# Find similar movies.

# Recommend them.

In [25]:
def recommend_item_item(user_id,top_n=10):
    watched=ratings[ratings.user_id==user_id]
    #print(watched)

    recommendations={}

    for movie in watched.movie_id:
        similar=similar_movies(movie,top_10)

        for sim_movie,score in similar.items():
            if sim_movie not in watched.movie_id.values:
                recommendations[sim_movie]=(
                    recommendations.get(sim_movie,0)
                    +score
                )


    recs=sorted(
        recommendations.items(),
        key=lambda x:x[1],
        reverse=True
    )


    return recs[:top_n]

# Matrix Factorization

In [26]:
from sklearn.decomposition import TruncatedSVD

In [27]:
interaction_matrix_filled = interaction_matrix.fillna(0)

In [28]:
svd = TruncatedSVD(
    n_components=200,
    random_state=42
)

user_factors = svd.fit_transform(
    interaction_matrix_filled
)

movie_factors = svd.components_

In [29]:
# from sklearn.decomposition import TruncatedSVD

# for k in [10, 20, 50, 100, 200]:
#     svd = TruncatedSVD(n_components=k, random_state=42)
#     svd.fit(interaction_matrix.fillna(0))

#     print(
#         f"{k} components: "
#         f"{svd.explained_variance_ratio_.sum():.4f}"
#     )

In [30]:
svd.explained_variance_ratio_.sum()

np.float64(0.80867891536662)

In [31]:
#predict ratings

predicted_ratings=np.dot(user_factors,movie_factors)

In [32]:
predicted_df = pd.DataFrame(
    predicted_ratings,
    index=interaction_matrix.index,
    columns=interaction_matrix.columns
)

In [33]:
print(predicted_df)

movie_id      1         2         3         4         5         6     \
user_id                                                                
1         5.103749  3.149637  2.860085  3.520720  2.383760  3.344657   
2         4.020787  0.067983  0.563765  0.144471  0.180772  0.292640   
3        -0.361274 -0.589274 -0.447987 -0.149591  0.222727 -0.179953   
4         0.211076 -0.467298  0.230919 -0.279176  0.189098  0.050607   
5         4.476509  1.958919  0.244592 -0.476236  0.704100  0.538959   
...            ...       ...       ...       ...       ...       ...   
939       0.350795 -0.664741 -0.437853  0.528549 -0.198681 -0.145976   
940       0.044894  0.253830  0.264414  2.523008  0.088330 -0.144067   
941       4.177208 -0.059712  0.443428 -0.627750 -0.344515 -0.256774   
942      -0.252772  0.547512  0.567960 -0.391515 -0.695541  0.032498   
943       0.202843  3.677450  1.034910  0.088336  0.443487 -0.010866   

movie_id      7         8         9         10    ...      1673

In [34]:
# SVD
#  ↓
# Predict ratings for ALL movies
#  ↓
# Remove movies already watched
#  ↓
# Recommend highest predicted ratings

In [35]:
def recommend_svd(user_id,top_n=10):
    user_prediction=predicted_df.loc[user_id]

    watched=ratings[ratings.user_id==user_id].movie_is.values

    recommendations=(
        user_predictions
        .drop(watched)
        .sort_values(ascending=False)
        .head(top_n)
    )

    return recommendations

# Evaluation

In [40]:
from surprise import Dataset #Simple Python Recommendation System Engine
from surprise import Reader
from surprise import SVD
from surprise.model_selection import train_test_split
from surprise import accuracy

In [49]:
reader=Reader(
    rating_scale=(1,5)
)

data=Dataset.load_from_df(
    ratings[['user_id','movie_id','rating']],
    reader
)

trainset,testset=train_test_split(
    data,
    test_size=0.2,
    random_state=42
)

In [55]:
model = SVD()

In [56]:
model.fit(trainset)

In [57]:
predictions = model.test(testset)

In [59]:
accuracy.rmse(predictions)
accuracy.mae(predictions)

RMSE: 0.9353
MAE:  0.7370


np.float64(0.7369744301378889)

# Saving models

In [66]:
user_similarity_df.to_parquet(
    "../data/processed/user_similarity.parquet"
)

movie_similarity_df.to_parquet(
    "../data/processed/movie_similarity.parquet"
)

predicted_df.to_parquet(
    "../data/processed/predicted_ratings.parquet"
)

In [70]:
import joblib

joblib.dump(
    model,
    "../data/processed/svd_model.pkl"
)

['../data/processed/svd_model.pkl']